# Radial Intensity Profiling of Micropatterned Colonies

**v1.0 — dual-segmenter, dual-mode pipeline.**

For each multichannel TIFF of a micropatterned colony this notebook:

1. Max-projects the Z-stack (or accepts an already-projected `(C, Y, X)` TIFF)
2. Gaussian-smooths + contrast-enhances each channel (`tapenade.global_contrast_enhancement`)
3. Segments nuclei on the DAPI channel — **StarDist** (default, fast on CPU) or **Cellpose-SAM** (better on dense/irregular nuclei; needs a GPU to be practical)
4. Finds the colony center as the center of mass of the nuclear mask
5. Computes radial intensity profiles in **two complementary modes**:
   - **Per-pixel** (classic): mean masked intensity per 1-px radial bin
   - **Per-nucleus** (recommended): one data point per nucleus (distance from center, mean intensity inside that nucleus) + LOWESS trend — this removes the noisy jump near r = 0 caused by tiny bins, and it is the biologically right readout for nuclear factors like SMAD2
6. Saves per-image plots + CSVs, cross-colony summary figures (including a publication-style multi-panel figure), and a PowerPoint report

**Channel assumption (edit in the parameters cell if different):**

| Index | Marker |
|---|---|
| C0 | ZO-1 |
| C1 | DAPI |
| C2 | SMAD2 |


## Imports

In [ ]:
import os
import re
import glob

import numpy as np
import pandas as pd
import tifffile
import matplotlib as mpl
import matplotlib.pyplot as plt

from scipy import ndimage
from scipy.ndimage import gaussian_filter, center_of_mass, zoom, distance_transform_edt
from skimage.measure import find_contours, regionprops
from skimage.segmentation import watershed
from skimage.feature import peak_local_max
from skimage.filters import threshold_otsu, unsharp_mask
import itertools
from tapenade.preprocessing import global_contrast_enhancement
from statsmodels.nonparametric.smoothers_lowess import lowess
from pptx import Presentation
from pptx.util import Inches


## Parameters — everything tunable lives here

In [ ]:
# === IMAGING (change for your microscope) ===
SCALE_UM_PER_PX = 0.5
CHANNEL_NAMES   = ['ZO-1', 'DAPI', 'SMAD2']
DAPI_IDX        = 1

# === SEGMENTATION BACKEND ===
# 'stardist' — fast on CPU (~2-3 s/image), great for typical hESC colonies
# 'cellpose' — Cellpose-SAM, more robust on dense/irregular nuclei, but slow
#              without a GPU (minutes per image on CPU)
SEGMENTER = 'stardist'

# --- StarDist settings ---
STARDIST_NORM_PERCENTILES = (1, 99.8)

# --- Cellpose settings (used only when SEGMENTER='cellpose') ---
# Missing nuclei?  lower CELLPROB (-1) / FLOW (0.3).  Extra junk? raise CELLPROB (+1) / MIN_SIZE.
CELLPOSE_DIAMETER_PX        = 35
CELLPOSE_FLOW_THRESHOLD     = 0.4
CELLPOSE_CELLPROB_THRESHOLD = 0.0
CELLPOSE_MIN_SIZE           = 15

# === PREPROCESSING ===
GAUSSIAN_SIGMA       = 1.0
CONTRAST_PERCENTILES = (0.5, 99.5)   # tapenade global contrast enhancement

# === ANALYSIS ===
MAX_RADIUS_UM      = 250    # truncation radius for the per-pixel profile
PLOT_MAX_UM        = 300    # x-axis extent
NORMALIZE_PROFILES = False  # False = keep absolute (a.u.) amplitudes so colonies can be
                            # compared to each other (recommended). True = each colony max -> 1.
BIN_WIDTH_UM       = 10     # radial bin width for per-nucleus summaries
LOWESS_FRAC        = 0.2    # LOWESS smoothing span (fraction of points per fit)
RATIO_TO_DAPI      = True   # also compute Cn/DAPI per nucleus (controls for density/section
                            # thickness; standard for nuclear factors like SMAD2)
BOOTSTRAP_N        = 1000   # colony-level bootstrap resamples for the CI band (>=4 colonies)

# === TAPENADE-PAPER EXTENSIONS (Gros et al., eLife 2026, doi:10.7554/eLife.107154) ===
DAPI_FIELD_NORM     = False  # re-normalize all channels by a masked-Gaussian DAPI field
                             # (2D version of their Fig. 5 optical-artifact correction)
FIELD_NORM_SIGMA_UM = 12     # field kernel ~ nucleus diameter (paper: 10-15 um optimal)
POSITIVE_FRACTION   = True   # Otsu per colony -> fraction of positive nuclei vs radius
                             # (their approach for sparse markers like FoxA2)
COEXPRESSION_PLOTS  = True   # per-nucleus pairwise co-expression histograms (their Fig. 5f)

# === SIZE & COVERAGE QC ===
EXPECTED_NUCLEUS_DIAMETER_UM = 12.0       # set manually for your cell type (hESC ~10-15 um)
NUCLEUS_SIZE_FILTER          = True       # reject segmented objects with implausible areas
NUCLEUS_AREA_RANGE_FACTOR    = (0.4, 2.5) # accepted area range, x expected nucleus area
NUCLEAR_PACKING_FRACTION     = 0.6        # max nuclear packing of colony area (Tapenade: <=61%)
MIN_BIN_COVERAGE             = 0.3        # radial bins with nuclei count < this x colony median
                                          # are treated as cell-free gaps and excluded
NUCLEUS_AREA_HARD_MAX_UM2    = 400.0      # absolute cap: larger objects are merged blobs of
                                          # many cells, always rejected (None = off)
DIRECTIONAL_SECTORS          = 8          # angular sectors for per-bin directionality QC
MIN_SECTOR_COVERAGE          = 0.75       # radial bin flagged if occupied-sector fraction below this
PIXEL_SIZE_OVERRIDES         = {}         # filename-substring -> µm/px for mixed-sampling batches,
                                          # e.g. {'72+ActA': 1.036} — read true values from CZI metadata;
                                          # size filters and distances are µm-based, so this must be right

# === SEGMENTATION RESCUE (keeps coarse/soft images in the dataset instead of excluding) ===
SEG_UPSAMPLE_TARGET_UM = 0.5   # auto-upsample segmenter input when the pixel size is coarser
                               # than 1.3x this, so nuclei reach the size the model expects;
                               # labels are mapped back to the native grid. (Rescued a
                               # 1.036 µm/px colony from 165 -> 888 kept nuclei.)
STARDIST_PROB_THRESH   = None  # e.g. 0.35 for a uniformly soft-focus batch (None = default)
SEG_UNSHARP            = False # unsharp-mask DAPI before segmentation (soft-focus rescue)
SPLIT_OVERSIZED        = True  # watershed-split objects above the hard area cap into
                               # nucleus-sized pieces (recovers merged multi-cell objects
                               # instead of discarding them)
SEG_OVERRIDES          = {}    # per-image segmentation settings by filename substring,
                               # e.g. {'72CAG+ActA_': {'unsharp': True, 'prob_thresh': 0.35}}
                               # for a single soft-focus colony, without touching the rest

# === GAP IMPUTATION (marble-in-jar; OFF by default) ===
# Fill empty (annulus x sector) cells by bootstrap-sampling nuclei from OTHER sectors of
# the SAME annulus (radial exchangeability). k per gap = jar-predicted count. Synthetic
# nuclei carry imputed=True: they stabilize the group curve and packing weights but are
# EXCLUDED from colony features and significance statistics.
IMPUTE_GAPS             = False
IMPUTE_BIN_UM           = 20    # annulus width of the imputation grid
IMPUTE_MIN_DONORS       = 30    # min real nuclei in the annulus donor pool
IMPUTE_MAX_GAP_FRACTION = 0.25  # refuse if more than this fraction of sectors are gaps
IMPUTE_SYMMETRY_CV_MAX  = 0.25  # refuse if donor sectors disagree (real directional biology)
IMPUTE_SEED             = 0     # reproducible draws

# === PLOT STYLE ===
CHANNEL_COLORS = {'ZO-1': '#00A087', 'DAPI': '#3C5488', 'SMAD2': '#E64B35'}  # npg palette
CHANNEL_CMAPS  = ['plasma', 'inferno', 'magma']

NATURE_RC = {
    'font.family': 'DejaVu Sans', 'font.size': 8,
    'axes.linewidth': 0.6, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.labelsize': 9, 'axes.titlesize': 10, 'axes.titleweight': 'bold',
    'xtick.labelsize': 7, 'ytick.labelsize': 7,
    'legend.fontsize': 7, 'legend.frameon': False,
    'pdf.fonttype': 42,
}


## Segmentation backend (lazy-loaded)

In [ ]:
_seg_fn = None

def get_segmenter():
    """Load the chosen segmentation model once; return a function img -> label image."""
    global _seg_fn
    if _seg_fn is not None:
        return _seg_fn

    if SEGMENTER == 'stardist':
        from stardist.models import StarDist2D
        from csbdeep.utils import normalize as _sd_normalize
        model = StarDist2D.from_pretrained('2D_versatile_fluo')

        def _segment(dapi):
            kw = {} if STARDIST_PROB_THRESH is None else {'prob_thresh': STARDIST_PROB_THRESH}
            labels, _ = model.predict_instances(
                _sd_normalize(dapi, *STARDIST_NORM_PERCENTILES, clip=True), **kw)
            return labels

    elif SEGMENTER == 'cellpose':
        from cellpose import models as _cp_models
        model = _cp_models.CellposeModel(gpu=False, pretrained_model='cpsam')

        def _segment(dapi):
            masks, _flows, _styles = model.eval(
                dapi,
                diameter=CELLPOSE_DIAMETER_PX,
                flow_threshold=CELLPOSE_FLOW_THRESHOLD,
                cellprob_threshold=CELLPOSE_CELLPROB_THRESHOLD,
                min_size=CELLPOSE_MIN_SIZE,
                normalize=True)
            return masks

    else:
        raise ValueError(f"SEGMENTER must be 'stardist' or 'cellpose', got {SEGMENTER!r}")

    _seg_fn = _segment
    return _seg_fn


## Helpers

In [ ]:
def save_figure(fig, path, show=True):
    fig.savefig(path, dpi=300, bbox_inches='tight')
    if show:
        plt.show()
    plt.close(fig)


def clean_label(name):
    """Readable label from filename; adjust the regex for your naming convention."""
    base = os.path.basename(name)
    for suffix in ('.tif', '.tiff'):
        base = base.replace(suffix, '')
    match = re.search(r'_(.*?)_488', base)
    return match.group(1).replace('_', ' ') if match else base


## Loading, preprocessing, segmentation

In [ ]:
def load_and_preprocess(path):
    """Accept 4D (Z, C, Y, X) or already-projected 3D (C, Y, X) TIFFs.
    Returns (img_enhanced, img_raw) as (C, Y, X) float32."""
    img = tifffile.imread(path)
    if img.ndim == 4:
        img = img.max(axis=0)
    elif img.ndim != 3:
        raise ValueError(f'Expected 3D (C,Y,X) or 4D (Z,C,Y,X) image, got shape {img.shape}')
    img = img.astype(np.float32)
    if img.shape[0] != len(CHANNEL_NAMES):
        raise ValueError(
            f'Expected {len(CHANNEL_NAMES)} channels {CHANNEL_NAMES}, got {img.shape[0]}')
    img_blur = np.stack([gaussian_filter(img[c], sigma=GAUSSIAN_SIGMA)
                         for c in range(img.shape[0])])
    img_enh = np.stack([
        global_contrast_enhancement(img_blur[c],
                                    perc_low=CONTRAST_PERCENTILES[0],
                                    perc_high=CONTRAST_PERCENTILES[1])
        for c in range(img.shape[0])])
    return img_enh, img


_DEFAULT_SEG_UNSHARP = SEG_UNSHARP
_DEFAULT_PROB_THRESH = STARDIST_PROB_THRESH

def seg_settings_for(name):
    """Per-image segmentation settings: first matching SEG_OVERRIDES substring, else defaults."""
    for pat, cfg in SEG_OVERRIDES.items():
        if pat in name:
            return (cfg.get('unsharp', _DEFAULT_SEG_UNSHARP),
                    cfg.get('prob_thresh', _DEFAULT_PROB_THRESH))
    return _DEFAULT_SEG_UNSHARP, _DEFAULT_PROB_THRESH


def split_oversized_labels(labels, scale_um):
    """Watershed-split objects larger than the hard area cap into nucleus-sized pieces.

    Seeds = local maxima of the distance transform spaced ~one nucleus radius apart, so a
    blob of N merged cells becomes ~N nuclei instead of one rejected object."""
    if not SPLIT_OVERSIZED:
        return labels
    exp_area = np.pi * (EXPECTED_NUCLEUS_DIAMETER_UM / 2.0) ** 2
    cap_um2 = NUCLEUS_AREA_HARD_MAX_UM2 or (NUCLEUS_AREA_RANGE_FACTOR[1] * exp_area)
    cap_px = cap_um2 / scale_um ** 2
    exp_r_px = max(2, int((EXPECTED_NUCLEUS_DIAMETER_UM / 2.0) / scale_um))
    ids, counts = np.unique(labels, return_counts=True)
    big = ids[(ids > 0) & (counts > cap_px)]
    if len(big) == 0:
        return labels
    out = labels.copy()
    next_id = int(labels.max()) + 1
    for lid in big:
        m = labels == lid
        ys, xs = np.where(m)
        y0, y1, x0, x1 = ys.min(), ys.max() + 1, xs.min(), xs.max() + 1
        sub = m[y0:y1, x0:x1]
        edt = distance_transform_edt(sub)
        pk = peak_local_max(edt, min_distance=exp_r_px, labels=sub, exclude_border=False)
        if len(pk) < 2:
            continue                      # cannot split; size filter will reject it
        markers = np.zeros(sub.shape, dtype=int)
        markers[tuple(pk.T)] = np.arange(1, len(pk) + 1)
        ws = watershed(-edt, markers, mask=sub)
        region = out[y0:y1, x0:x1]
        region[sub] = 0
        for wid in range(1, int(ws.max()) + 1):
            region[ws == wid] = next_id
            next_id += 1
    return out


def segment_nuclei(img_enh):
    """Segment nuclei on the DAPI channel. Returns (labels, mask, center_xy_float).

    Rescue logic: coarse-sampled images (px > 1.3x SEG_UPSAMPLE_TARGET_UM) are upsampled
    before segmentation and the labels mapped back — nuclei that are only ~11 px across at
    1 µm/px merge irrecoverably otherwise. SEG_UNSHARP sharpens soft-focus DAPI first."""
    dapi = img_enh[DAPI_IDX]
    if SEG_UNSHARP:
        dapi = unsharp_mask(dapi, radius=2, amount=1.5)
    factor = 1
    if SEG_UPSAMPLE_TARGET_UM and SCALE_UM_PER_PX > 1.3 * SEG_UPSAMPLE_TARGET_UM:
        factor = int(round(SCALE_UM_PER_PX / SEG_UPSAMPLE_TARGET_UM))
        dapi = zoom(dapi, factor, order=1)
    labels = get_segmenter()(dapi)
    if factor > 1:
        labels = labels[::factor, ::factor]
    labels = split_oversized_labels(labels, SCALE_UM_PER_PX)
    mask = labels > 0
    if not mask.any():
        raise RuntimeError('No nuclei found — check DAPI channel / segmenter settings')
    cy, cx = center_of_mass(mask)
    return labels, mask.astype(np.uint8), (cx, cy)


## Mode 1 — per-pixel radial profile (classic)

Mean masked intensity per 1-px radial bin. The truncation past `MAX_RADIUS_UM` keeps the
original behaviour of the V3 pipeline: beyond the cutoff only values at or below the
regional minimum are kept, the rest become NaN and are interpolated for display.


In [ ]:
def radial_profile_masked(data, center, mask):
    y, x = np.indices(data.shape)
    r = np.sqrt((x - center[0])**2 + (y - center[1])**2).astype(int)
    valid = mask.astype(bool)
    counts = np.bincount(r[valid].ravel())
    sums   = np.bincount(r[valid].ravel(), data[valid].ravel())
    profile = np.where(counts > 0, sums / np.maximum(counts, 1), np.nan)
    radii_um = np.arange(len(profile)) * SCALE_UM_PER_PX
    return radii_um, profile


def truncate_profile(radii_um, profile):
    """Original V3 truncation: past MAX_RADIUS_UM keep only values <= the regional
    minimum; everything else becomes NaN (interpolated later for plotting)."""
    profile = profile.copy()
    post = radii_um > MAX_RADIUS_UM
    if post.any():
        vals = profile[post]
        if np.any(np.isfinite(vals)):
            vals = np.where(vals > np.nanmin(vals), np.nan, vals)
            profile[post] = vals
    return profile


## Mode 2 — per-nucleus profile (recommended)

One data point per nucleus: distance of the nucleus centroid from the colony center vs the
mean enhanced intensity inside that nucleus. Then a LOWESS trend per colony.

Why this is better than per-pixel binning:

- **No start-up jump.** The innermost 1-px rings contain only a handful of pixels, so the
  classic profile is wildly noisy near r = 0. Nuclei are natural averaging units and there
  simply are no nuclei at r ≈ 0, so the curve starts where data exists.
- **Biologically correct for nuclear readouts.** SMAD2 signaling activity *is* nuclear
  intensity per nucleus — this is how the micropattern papers (Etoc 2016, Chhabra 2019)
  quantify it.
- **Statistics for free.** n = nuclei per bin → honest SEM bands.


In [ ]:
def per_nucleus_table(img_enh, labels, center):
    """One row per nucleus: position, distance, area, shape metrics, mean intensity/channel.

    Shape metrics follow Gros et al. (eLife 2026): nuclei as proxies for cell deformation.
    cos2_radial = squared cosine between the nucleus major axis and the radial direction
    (1 = radially aligned, 0 = circumferential, 0.5 = random baseline in 2D)."""
    ids = np.arange(1, int(labels.max()) + 1)
    ones = np.ones(labels.shape, dtype=np.float32)
    coms = ndimage.center_of_mass(ones, labels, ids)
    areas_px = ndimage.sum_labels(ones, labels, ids)
    ys = np.array([p[0] for p in coms]); xs = np.array([p[1] for p in coms])
    dist_um = np.sqrt((xs - center[0])**2 + (ys - center[1])**2) * SCALE_UM_PER_PX

    ecc = np.full(len(ids), np.nan)
    cos2 = np.full(len(ids), np.nan)
    major_um = np.full(len(ids), np.nan)
    ori = np.full(len(ids), np.nan)
    for p in regionprops(labels):
        j = p.label - 1
        if j >= len(ids):
            continue
        ecc[j] = p.eccentricity
        major_um[j] = p.axis_major_length * SCALE_UM_PER_PX
        ori[j] = p.orientation
        u = np.array([np.sin(p.orientation), np.cos(p.orientation)])   # major axis (x, y)
        v = np.array([xs[j] - center[0], ys[j] - center[1]])
        n = np.linalg.norm(v)
        if n > 0:
            cos2[j] = float((u @ (v / n)) ** 2)

    out = {
        'nucleus_id': ids, 'x_px': xs, 'y_px': ys,
        'distance_um': dist_um,
        'area_um2': areas_px * SCALE_UM_PER_PX**2,
        'eccentricity': ecc, 'major_axis_um': major_um, 'cos2_radial': cos2,
        'orientation_rad': ori,
    }
    for ch in range(img_enh.shape[0]):
        out[f'C{ch}'] = ndimage.mean(img_enh[ch], labels, ids)
    df = pd.DataFrame(out)
    q99 = df['distance_um'].quantile(0.99)
    df['rel_distance'] = df['distance_um'] / q99 if q99 else np.nan
    if RATIO_TO_DAPI:
        dapi = df[f'C{DAPI_IDX}'].replace(0, np.nan)
        for ch in range(img_enh.shape[0]):
            if ch != DAPI_IDX:
                df[f'C{ch}_over_DAPI'] = df[f'C{ch}'] / dapi
    return df


## Gap imputation (marble-in-jar)

A gap wedge at radius r borrows datapoints ONLY from other sectors at the same r
(radial exchangeability). The jar model sets how many to draw. Refused when the donor
pool is thin, the gap is too large, or donor sectors genuinely disagree (directional
biology must stay visible). Synthetic rows are flagged `imputed=True` and never enter
features or statistics.


In [ ]:
def impute_gaps(nuc_df, center):
    """Fill gap (annulus x sector) cells from same-annulus donors. Returns (df, n_added)."""
    real = nuc_df[nuc_df['size_ok'] & (nuc_df['distance_um'] <= MAX_RADIUS_UM)]
    if len(real) < 50:
        return nuc_df, 0
    rng_i = np.random.default_rng(IMPUTE_SEED)
    a_nuc = np.pi * (EXPECTED_NUCLEUS_DIAMETER_UM / 2.0) ** 2
    ang = np.arctan2(real['y_px'] - center[1], real['x_px'] - center[0])
    sec = np.clip(((ang + np.pi) / (2 * np.pi) * DIRECTIONAL_SECTORS).astype(int),
                  0, DIRECTIONAL_SECTORS - 1)
    ann = (real['distance_um'] // IMPUTE_BIN_UM).astype(int)
    q99 = real['distance_um'].quantile(0.99)
    new_rows = []
    next_id = int(nuc_df['nucleus_id'].max()) + 1
    for bi in range(int(MAX_RADIUS_UM // IMPUTE_BIN_UM)):
        r0, r1 = bi * IMPUTE_BIN_UM, (bi + 1) * IMPUTE_BIN_UM
        pred_cell = (NUCLEAR_PACKING_FRACTION * np.pi * (r1**2 - r0**2)
                     / DIRECTIONAL_SECTORS / a_nuc)
        if pred_cell < 2:
            continue                                   # inner rings too small to judge
        in_ann = real[ann == bi]
        cnts = pd.Series(sec[ann == bi]).value_counts()
        cnts = cnts.reindex(range(DIRECTIONAL_SECTORS), fill_value=0)
        gaps = cnts[cnts < 0.35 * pred_cell].index.tolist()
        donor_secs = cnts[cnts >= 0.6 * pred_cell].index.tolist()
        donors = in_ann[np.isin(sec[ann == bi], donor_secs)]
        if (not gaps or len(donors) < IMPUTE_MIN_DONORS
                or len(gaps) > IMPUTE_MAX_GAP_FRACTION * DIRECTIONAL_SECTORS
                or len(donor_secs) < 4):
            continue
        # symmetry check: donor sectors must agree on every channel
        ok_sym = True
        dsec = sec[ann == bi][np.isin(sec[ann == bi], donor_secs)]
        for ch in range(len(CHANNEL_NAMES)):
            m = donors.groupby(dsec.values)[f'C{ch}'].mean()
            if len(m) > 2 and m.mean() and (m.std() / m.mean()) > IMPUTE_SYMMETRY_CV_MAX:
                ok_sym = False
                break
        if not ok_sym:
            continue
        for s in gaps:
            k = int(round(pred_cell - cnts[s]))
            if k <= 0:
                continue
            picks = donors.sample(k, replace=True,
                                  random_state=int(rng_i.integers(0, 2**31)))
            th0 = -np.pi + s * 2 * np.pi / DIRECTIONAL_SECTORS
            for _, row in picks.iterrows():
                r_new = np.sqrt(rng_i.uniform(r0**2, r1**2))
                t_new = rng_i.uniform(th0, th0 + 2 * np.pi / DIRECTIONAL_SECTORS)
                nr = row.copy()
                nr['nucleus_id'] = next_id
                next_id += 1
                nr['x_px'] = center[0] + (r_new / SCALE_UM_PER_PX) * np.cos(t_new)
                nr['y_px'] = center[1] + (r_new / SCALE_UM_PER_PX) * np.sin(t_new)
                nr['distance_um'] = r_new
                nr['rel_distance'] = r_new / q99 if q99 else np.nan
                nr['imputed'] = True
                new_rows.append(nr)
    if not new_rows:
        return nuc_df, 0
    return pd.concat([nuc_df, pd.DataFrame(new_rows)], ignore_index=True), len(new_rows)


## Optional: DAPI-field intensity re-normalization (Gros et al. 2026)

The Tapenade paper corrects optical/illumination artifacts by dividing each channel by a
**masked-Gaussian coarse-grained map of the ubiquitous nuclear stain** (their Fig. 5). The 2D
version here corrects uneven illumination across the colony. After normalization the DAPI
channel should be nearly flat — a built-in QC that the correction worked. Off by default
(`DAPI_FIELD_NORM`) to preserve backward comparability.


In [ ]:
def dapi_field_normalize(img_enh, mask):
    """Divide all channels by a masked-Gaussian DAPI field (2D of Gros et al. Fig. 5)."""
    sigma_px = FIELD_NORM_SIGMA_UM / SCALE_UM_PER_PX
    m = mask.astype(np.float32)
    den = gaussian_filter(m, sigma_px)
    num = gaussian_filter(img_enh[DAPI_IDX] * m, sigma_px)
    field = np.where(den > 1e-3, num / np.maximum(den, 1e-3), np.nan)
    ref = np.nanmean(field[mask.astype(bool)])
    field = np.where(np.isfinite(field) & (field > 0), field, ref)
    return img_enh * (ref / field)[None, :, :]


## Per-image analysis

In [ ]:
_DEFAULT_SCALE_UM_PER_PX = SCALE_UM_PER_PX

def pixel_size_for(name):
    """Per-image pixel size: first matching PIXEL_SIZE_OVERRIDES substring, else default."""
    for pat, um in PIXEL_SIZE_OVERRIDES.items():
        if pat in name:
            return float(um)
    return _DEFAULT_SCALE_UM_PER_PX


def full_analysis(path, out_root, plots_root):
    img_name = os.path.basename(path)
    global SCALE_UM_PER_PX, SEG_UNSHARP, STARDIST_PROB_THRESH
    SCALE_UM_PER_PX = pixel_size_for(img_name)
    SEG_UNSHARP, STARDIST_PROB_THRESH = seg_settings_for(img_name)
    stem = re.sub(r'\.tiff?$', '', img_name, flags=re.IGNORECASE)
    plot_dir = os.path.join(plots_root, stem)
    os.makedirs(plot_dir, exist_ok=True)

    img_enh, img_raw = load_and_preprocess(path)
    labels, mask, center = segment_nuclei(img_enh)
    if DAPI_FIELD_NORM:
        img_enh = dapi_field_normalize(img_enh, mask)
    n_nuclei = int(labels.max())

    # Size-based QC (objective 3): build the nucleus table first, reject implausible areas,
    # and remove rejected nuclei from the pixel mask so both modes see the same cells.
    nuc_df = per_nucleus_table(img_enh, labels, center)
    exp_area_um2 = np.pi * (EXPECTED_NUCLEUS_DIAMETER_UM / 2.0) ** 2
    if NUCLEUS_SIZE_FILTER:
        lo_f, hi_f = NUCLEUS_AREA_RANGE_FACTOR
        ok = nuc_df['area_um2'].between(lo_f * exp_area_um2, hi_f * exp_area_um2)
    else:
        ok = pd.Series(True, index=nuc_df.index)
    if NUCLEUS_AREA_HARD_MAX_UM2 is not None:
        ok &= nuc_df['area_um2'] <= NUCLEUS_AREA_HARD_MAX_UM2
    nuc_df['size_ok'] = ok
    n_rejected = int((~nuc_df['size_ok']).sum())
    nuc_df['imputed'] = False
    n_imputed = 0
    if IMPUTE_GAPS:
        nuc_df, n_imputed = impute_gaps(nuc_df, center)
    if n_rejected:
        keep_ids = nuc_df.loc[nuc_df['size_ok'], 'nucleus_id'].values
        mask = np.isin(labels, keep_ids).astype(np.uint8)

    H, W = img_enh[DAPI_IDX].shape
    extent = [-W/2*SCALE_UM_PER_PX, W/2*SCALE_UM_PER_PX,
              -H/2*SCALE_UM_PER_PX, H/2*SCALE_UM_PER_PX]

    # --- Max projections ---
    fig, axs = plt.subplots(1, img_raw.shape[0], figsize=(4*img_raw.shape[0], 4))
    for i in range(img_raw.shape[0]):
        axs[i].imshow(img_raw[i], cmap='gray')
        axs[i].set_title(f'Max Projection: {CHANNEL_NAMES[i]}')
        axs[i].axis('off')
    save_figure(fig, os.path.join(plot_dir, 'max_projection.png'))

    # --- Segmentation overlay ---
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(img_enh[DAPI_IDX], cmap='gray', extent=extent, origin='lower')
    for cnt in find_contours(labels, 0.5):
        ax.plot((cnt[:, 1] - W/2)*SCALE_UM_PER_PX,
                (cnt[:, 0] - H/2)*SCALE_UM_PER_PX, lw=1, color='lime')
    ax.plot((center[0] - W/2)*SCALE_UM_PER_PX, (center[1] - H/2)*SCALE_UM_PER_PX,
            'y+', markersize=20, mew=3)
    ax.set_title(f'DAPI + {SEGMENTER} ({n_nuclei} nuclei)')
    ax.axis('equal')
    save_figure(fig, os.path.join(plot_dir, 'segmentation.png'))

    # --- Heatmaps ---
    fig, axs = plt.subplots(1, img_enh.shape[0], figsize=(4*img_enh.shape[0], 4))
    for i in range(img_enh.shape[0]):
        axs[i].imshow(img_enh[i], cmap=CHANNEL_CMAPS[i], extent=extent, origin='lower')
        axs[i].set_title(CHANNEL_NAMES[i])
        axs[i].axis('equal')
    plt.tight_layout()
    save_figure(fig, os.path.join(plot_dir, 'heatmaps.png'))

    # --- Mode 1: per-pixel profiles ---
    ylabel = 'Normalized Intensity' if NORMALIZE_PROFILES else 'Intensity (a.u.)'
    pix_profiles, full_radius = {}, None
    for c in range(img_enh.shape[0]):
        radii, prof = radial_profile_masked(img_enh[c], center, mask)
        if NORMALIZE_PROFILES:
            peak = np.nanmax(prof)
            if peak and peak > 0:
                prof = prof / peak
        prof = truncate_profile(radii, prof)
        if NORMALIZE_PROFILES:
            peak = np.nanmax(prof)
            if peak and peak > 0:
                prof = prof / peak
        pix_profiles[f'C{c}'] = prof
        if full_radius is None:
            full_radius = radii

        name = CHANNEL_NAMES[c]
        fig, ax = plt.subplots()
        ax.plot(radii, pd.Series(prof).interpolate(limit_direction='both'),
                lw=2, color=CHANNEL_COLORS[name])
        ax.set_xlim(0, PLOT_MAX_UM)
        ax.set_xlabel('Distance from Center (µm)')
        ax.set_ylabel(ylabel)
        ax.set_title(f'{name} Radial Profile (per-pixel)')
        save_figure(fig, os.path.join(plot_dir, f'profile_C{c}.png'))

    pix_df = pd.DataFrame(pix_profiles)
    pix_df.insert(0, 'Radius', full_radius)
    pix_df.to_csv(os.path.join(out_root, f'{stem}_radial.csv'), index=False)

    fig, ax = plt.subplots(figsize=(6, 4))
    for c in range(img_enh.shape[0]):
        name = CHANNEL_NAMES[c]
        ax.plot(full_radius,
                pd.Series(pix_profiles[f'C{c}']).interpolate(limit_direction='both'),
                lw=2, label=name, color=CHANNEL_COLORS[name])
    ax.set_xlim(0, PLOT_MAX_UM)
    ax.set_xlabel('Distance from Center (µm)')
    ax.set_ylabel(ylabel)
    ax.set_title('Combined Radial Profiles (per-pixel)')
    ax.legend()
    save_figure(fig, os.path.join(plot_dir, 'combined_profiles.png'))

    # --- Mode 2: per-nucleus + LOWESS (size-filtered) ---
    nuc_df.to_csv(os.path.join(out_root, f'{stem}_nuclei.csv'), index=False)

    nuc_plot = nuc_df[nuc_df['size_ok']]
    fig, axs = plt.subplots(1, img_enh.shape[0], figsize=(5*img_enh.shape[0], 4))
    for c in range(img_enh.shape[0]):
        name = CHANNEL_NAMES[c]
        axs[c].scatter(nuc_plot['distance_um'], nuc_plot[f'C{c}'],
                       s=4, alpha=0.3, color='gray')
        if len(nuc_plot) >= 30:
            sm = lowess(nuc_plot[f'C{c}'].values, nuc_plot['distance_um'].values,
                        frac=LOWESS_FRAC, return_sorted=True)
            axs[c].plot(sm[:, 0], sm[:, 1], color=CHANNEL_COLORS[name], lw=2.5)
        axs[c].set_xlim(0, PLOT_MAX_UM)
        axs[c].set_xlabel('Distance from Center (µm)')
        axs[c].set_ylabel(f'{name} (a.u.)')
        axs[c].set_title(f'{name} per-nucleus ({len(nuc_plot)} kept, {n_rejected} size-rejected)')
    plt.tight_layout()
    save_figure(fig, os.path.join(plot_dir, 'per_nucleus_profiles.png'))

    # --- Per-bin technical segmentation coverage (weights for group averaging) ---
    sigm = img_enh[DAPI_IDX] > threshold_otsu(img_enh[DAPI_IDX])
    yy, xx = np.indices(sigm.shape)
    r_um_pix = np.sqrt((xx - center[0])**2 + (yy - center[1])**2) * SCALE_UM_PER_PX
    kmax = int(PLOT_MAX_UM // BIN_WIDTH_UM) + 1
    kk = np.minimum((r_um_pix // BIN_WIDTH_UM).astype(int), kmax)
    dapi_px = np.bincount(kk.ravel(), sigm.ravel().astype(float), minlength=kmax + 1)
    seg_px  = np.bincount(kk.ravel(), (mask > 0).ravel().astype(float), minlength=kmax + 1)
    with np.errstate(divide='ignore', invalid='ignore'):
        coverage = np.where(dapi_px * SCALE_UM_PER_PX**2 > 500, seg_px / dapi_px, np.nan)
    cov_df = pd.DataFrame({
        'bin': np.arange(kmax) * BIN_WIDTH_UM + BIN_WIDTH_UM / 2,
        'dapi_area_um2': (dapi_px * SCALE_UM_PER_PX**2)[:kmax],
        'seg_area_um2':  (seg_px * SCALE_UM_PER_PX**2)[:kmax],
        'coverage': coverage[:kmax],
    })

    return (pix_df.assign(Image=img_name), nuc_df.assign(Image=img_name),
            cov_df.assign(Image=img_name), plot_dir)


## Quantitative feature extraction

Curves are for looking at; statistics need numbers. For each colony this extracts:

- **peak_radius_um / peak_value** — where and how high the LOWESS-smoothed profile peaks
- **halfmax_radius_um** — radius where the profile first crosses 50% of its dynamic range
  (a robust "boundary position" for edge-restricted signals like SMAD2)
- **center_edge_ratio** — mean signal in the inner 25% of the colony radius vs the outer 25%
  (single number that captures "edge-high" vs "center-high" patterning)
- **angular_cv** — coefficient of variation of the signal across 12 angular sectors
  (QC: high values flag off-center or asymmetric colonies whose radial average is suspect)


In [ ]:
def extract_features(nuc_df, channel_col, colony_radius_um=None):
    """Per-colony summary features from the per-nucleus table for one channel."""
    d = nuc_df.dropna(subset=[channel_col]).sort_values('distance_um')
    if len(d) < 30:
        return None
    r_max = colony_radius_um or d['distance_um'].quantile(0.99)
    sm = lowess(d[channel_col].values, d['distance_um'].values,
                frac=LOWESS_FRAC, return_sorted=True)
    r_s, v_s = sm[:, 0], sm[:, 1]
    in_range = r_s <= r_max
    r_s, v_s = r_s[in_range], v_s[in_range]
    if len(v_s) < 10:
        return None

    i_pk = int(np.nanargmax(v_s))
    v_lo, v_hi = np.nanmin(v_s), np.nanmax(v_s)
    half = v_lo + 0.5 * (v_hi - v_lo)
    above = np.where(v_s >= half)[0]
    halfmax_r = float(r_s[above[0]]) if len(above) else np.nan

    inner = d[d['distance_um'] <= 0.25 * r_max][channel_col].mean()
    outer = d[d['distance_um'] >= 0.75 * r_max][channel_col].mean()

    return {
        'n_nuclei': len(d),
        'colony_radius_um': float(r_max),
        'peak_radius_um': float(r_s[i_pk]),
        'peak_value': float(v_s[i_pk]),
        'halfmax_radius_um': halfmax_r,
        'center_edge_ratio': float(inner / outer) if outer else np.nan,
    }


def angular_cv(nuc_df, channel_col, center_xy=None, n_sectors=12):
    """QC: coefficient of variation of per-sector mean signal. High = asymmetric colony.
    Uses nucleus positions relative to the distance-weighted centroid of nuclei."""
    d = nuc_df.dropna(subset=[channel_col])
    if len(d) < n_sectors * 3 or 'x_px' not in d.columns:
        return np.nan
    ang = np.arctan2(d['y_px'] - d['y_px'].mean(), d['x_px'] - d['x_px'].mean())
    sector = ((ang + np.pi) / (2 * np.pi) * n_sectors).astype(int).clip(0, n_sectors - 1)
    means = d.groupby(sector)[channel_col].mean()
    return float(means.std() / means.mean()) if means.mean() else np.nan


## PowerPoint summary slide

In [ ]:
def add_image_summary_slide(prs, plot_dir, image_name):
    slide = prs.slides.add_slide(prs.slide_layouts[5])
    tb = slide.shapes.add_textbox(Inches(0.3), Inches(0.05), Inches(12), Inches(0.5))
    tb.text_frame.text = f'Results for {image_name}'
    layout = {
        'combined_profiles.png':   (8.0, 0.5),
        'profile_C0.png':          (0.3, 1.2),
        'profile_C1.png':          (3.0, 1.2),
        'profile_C2.png':          (5.7, 1.2),
        'max_projection.png':      (0.3, 3.9),
        'segmentation.png':        (3.0, 3.9),
        'per_nucleus_profiles.png':(5.7, 3.9),
    }
    for fname, (x, y) in layout.items():
        p = os.path.join(plot_dir, fname)
        if os.path.exists(p):
            slide.shapes.add_picture(p, Inches(x), Inches(y),
                                     width=Inches(2.5), height=Inches(2.5))


## Troubleshooting: QC report, provenance, and interactive inspection

Three layers, in the spirit of the Tapenade paper's tooling:

1. **`summary/qc_report.html`** — written automatically after every batch: thumbnail
   gallery (segmentation + profiles per image), nuclei counts with outlier flags,
   angular-asymmetry flags, failures, and the exact parameters — one file you can open in
   any browser months later to see what happened.
2. **`summary/run_parameters.json`** — every parameter + package version + timestamp, so
   two runs can always be diffed.
3. **`inspect_in_napari(path)`** (next section) — one call opens any image with its
   segmentation in napari for interactive GUI troubleshooting.
4. **Co-expression quadrants as a tuning readout** (`coexpression_per_nucleus.png`,
   quadrant percentages annotated). Reading DAPI (x) vs a marker like SMAD2 (y):
   - Healthy segmentation + staining puts nearly all nuclei **right** of the DAPI Otsu
     line; the marker axis then splits real biology into +/-.
   - A large lower-left cloud = DAPI-dim junk got segmented → tighten
     `NUCLEUS_SIZE_FILTER` / `NUCLEUS_AREA_RANGE_FACTOR`, or raise
     `STARDIST_NORM_PERCENTILES`.
   - Population squashed against the marker floor with little dynamic range → raise the
     upper `CONTRAST_PERCENTILES` (99.5 → 99.9) — or the issue is acquisition-side
     (exposure/antibody), which no pipeline parameter can fix honestly.
   - Moving cells to the upper-right quadrant is an **assay** goal (stronger specific
     signal); pipeline parameters should only be tuned to *represent* the population
     faithfully, not to push cells across thresholds.


In [ ]:
def dump_run_parameters(summary_root):
    """Write all ALL-CAPS parameters + package versions + timestamp to JSON."""
    import sys, importlib, datetime, json as _json
    params = {k: v for k, v in globals().items()
              if k.isupper() and isinstance(v, (int, float, str, bool, tuple, list, dict))}
    versions = {}
    for pkg in ('numpy', 'pandas', 'scipy', 'skimage', 'tifffile', 'stardist',
                'cellpose', 'tapenade', 'statsmodels'):
        try:
            versions[pkg] = getattr(importlib.import_module(pkg), '__version__', '?')
        except Exception:
            versions[pkg] = 'not installed'
    payload = {'timestamp': datetime.datetime.now().isoformat(timespec='seconds'),
               'python': sys.version.split()[0],
               'parameters': params, 'package_versions': versions}
    with open(os.path.join(summary_root, 'run_parameters.json'), 'w') as f:
        _json.dump(payload, f, indent=2, default=str)


def _thumb_b64(png_path, width=460, quality=70):
    from PIL import Image
    import io, base64
    im = Image.open(png_path)
    im.thumbnail((width, width * 10))
    buf = io.BytesIO()
    im.convert('RGB').save(buf, format='JPEG', quality=quality)
    return base64.b64encode(buf.getvalue()).decode()


def build_qc_report(base_dir):
    """Self-contained HTML QC report for a finished (or old) batch run."""
    import html as _html, datetime
    summary_root = os.path.join(base_dir, 'summary')
    plots_root   = os.path.join(base_dir, 'plots')
    out_root     = os.path.join(base_dir, 'radial_outputs')

    feats = None
    fp = os.path.join(summary_root, 'colony_features.csv')
    if os.path.exists(fp):
        feats = pd.read_csv(fp)

    failed = {}
    flog = os.path.join(summary_root, 'failed_images.log')
    if os.path.exists(flog):
        for line in open(flog):
            if line.strip():
                nm, _, err = line.partition('\t')
                failed[nm.strip()] = err.strip()

    stems = sorted(os.listdir(plots_root)) if os.path.isdir(plots_root) else []
    stems = [s for s in stems if os.path.isdir(os.path.join(plots_root, s))]

    counts = {}
    for s in stems:
        ncsv = os.path.join(out_root, f'{s}_nuclei.csv')
        if os.path.exists(ncsv):
            counts[s] = len(pd.read_csv(ncsv))
    median_n = float(np.median(list(counts.values()))) if counts else 0

    rows_html = []
    for s in stems:
        n = counts.get(s)
        flags = []
        if n is not None and median_n and (n < 0.5 * median_n or n > 1.5 * median_n):
            flags.append(f'⚠ nuclei count outlier (batch median {median_n:.0f})')
        if feats is not None:
            fr = feats[feats['Image'].str.replace(r'\.tiff?$', '', regex=True) == s]
            if len(fr):
                if fr['angular_cv'].max() > 0.25:
                    flags.append(f"⚠ angular asymmetry (CV {fr['angular_cv'].max():.2f})")
                r0 = fr.iloc[0]
                if 'occupancy_vs_capacity' in fr.columns and np.isfinite(r0['occupancy_vs_capacity']):
                    if r0['occupancy_vs_capacity'] > 1.2:
                        flags.append(f"⚠ {r0['occupancy_vs_capacity']:.2f}x size-based capacity — over-segmentation?")
                    elif r0['occupancy_vs_capacity'] < 0.3:
                        flags.append(f"⚠ only {r0['occupancy_vs_capacity']:.2f}x capacity — sparse/under-segmentation?")
                if 'dapi_bin_cv' in fr.columns and np.isfinite(r0['dapi_bin_cv']) and r0['dapi_bin_cv'] > 0.25:
                    flags.append(f"⚠ DAPI non-uniform (CV {r0['dapi_bin_cv']:.2f}) — cell-free zones?")
                if 'n_rejected_size' in fr.columns and r0['n_rejected_size'] > 0:
                    flags.append(f"{int(r0['n_rejected_size'])} nuclei size-rejected")
                if 'pct_rejected_size' in fr.columns and np.isfinite(r0.get('pct_rejected_size', np.nan)) and r0['pct_rejected_size'] > 15:
                    flags.append(f"⚠ {r0['pct_rejected_size']:.0f}% size-rejected — bad segmentation?")
                if 'area_cv' in fr.columns and np.isfinite(r0.get('area_cv', np.nan)) and r0['area_cv'] > 0.6:
                    flags.append(f"⚠ nucleus-area CV {r0['area_cv']:.2f} — merged/fragmented objects?")
                if 'n_directional_gap_bins' in fr.columns and np.isfinite(r0.get('n_directional_gap_bins', np.nan)) and r0['n_directional_gap_bins'] >= 3:
                    flags.append(f"⚠ {int(r0['n_directional_gap_bins'])} radial bins missing angular sectors")
        thumbs = ''
        for png in ('segmentation.png', 'combined_profiles.png', 'per_nucleus_profiles.png'):
            p = os.path.join(plots_root, s, png)
            if os.path.exists(p):
                thumbs += f'<img src="data:image/jpeg;base64,{_thumb_b64(p)}" style="max-width:300px;margin:2px">'
        flag_html = '<br>'.join(flags) if flags else '<span style="color:#00A087">OK</span>'
        rows_html.append(
            f'<tr><td><b>{_html.escape(s)}</b><br>{n if n is not None else "?"} nuclei<br>'
            f'{flag_html}</td><td>{thumbs}</td></tr>')

    for nm, err in failed.items():
        rows_html.append(
            f'<tr><td style="color:#E64B35"><b>{_html.escape(nm)}</b><br>FAILED</td>'
            f'<td><code>{_html.escape(err)}</code></td></tr>')

    top_figs = ''
    for png in ('publication_figure.png', 'positive_fraction_profiles.png',
                'nuclear_morphometrics.png', 'coexpression_per_nucleus.png'):
        p = os.path.join(summary_root, png)
        if os.path.exists(p):
            top_figs += f'<h3>{png}</h3><img src="data:image/jpeg;base64,{_thumb_b64(p, width=900)}" style="max-width:100%">'

    params_html = ''
    pj = os.path.join(summary_root, 'run_parameters.json')
    if os.path.exists(pj):
        params_html = f'<h2>Run parameters</h2><pre style="background:#f6f6f6;padding:10px;overflow-x:auto">{_html.escape(open(pj).read())}</pre>'

    doc = f"""<!doctype html><html><head><meta charset="utf-8">
<title>QC report — radial intensity pipeline</title>
<style>body{{font-family:sans-serif;max-width:1100px;margin:20px auto;padding:0 12px}}
table{{border-collapse:collapse;width:100%}}td{{border:1px solid #ddd;padding:6px;vertical-align:top}}
h1,h2,h3{{color:#3C5488}}</style></head><body>
<h1>Radial intensity pipeline — QC report</h1>
<p>Generated {datetime.datetime.now().isoformat(timespec='seconds')} ·
{len(stems)} image(s) processed · {len(failed)} failed · segmenter: {SEGMENTER}</p>
<h2>Summary figures</h2>{top_figs}
<h2>Per-image QC</h2><table>{''.join(rows_html)}</table>
{params_html}
</body></html>"""
    out = os.path.join(summary_root, 'qc_report.html')
    with open(out, 'w') as f:
        f.write(doc)
    return out


## Batch run

Point `BASE_DIR` at your folder of `.tif` files and run. Outputs land under `BASE_DIR`:

```
radial_outputs/   <image>_radial.csv (per-pixel) + <image>_nuclei.csv (per-nucleus)
plots/<image>/    per-image PNGs
summary/          combined CSVs, cross-colony figures, publication figure (PNG+PDF),
                  failed_images.log
ppt_reports/      batch_report.pptx
```


In [ ]:
# CHANGE THIS to point at your folder of TIFF images
BASE_DIR = 'PATH_TO_YOUR_TIFF_FOLDER'

if BASE_DIR == 'PATH_TO_YOUR_TIFF_FOLDER':
    print('⚠ Set BASE_DIR above to your TIFF folder path, then re-run this cell.')
else:
    out_root     = os.path.join(BASE_DIR, 'radial_outputs')
    plots_root   = os.path.join(BASE_DIR, 'plots')
    ppt_root     = os.path.join(BASE_DIR, 'ppt_reports')
    summary_root = os.path.join(BASE_DIR, 'summary')
    for d in (out_root, plots_root, ppt_root, summary_root):
        os.makedirs(d, exist_ok=True)

    dump_run_parameters(summary_root)

    tifs = sorted(glob.glob(os.path.join(BASE_DIR, '*.tif')) +
                  glob.glob(os.path.join(BASE_DIR, '*.tiff')))
    print(f'Found {len(tifs)} TIFF file(s); segmenter = {SEGMENTER}')

    failure_log = os.path.join(summary_root, 'failed_images.log')
    open(failure_log, 'w').close()

    pix_all, nuc_all, cov_all = [], [], []
    batch_ppt = Presentation()

    for fpath in tifs:
        fname = os.path.basename(fpath)
        try:
            pix_df, nuc_df, cov_df, plot_dir = full_analysis(fpath, out_root, plots_root)
            pix_all.append(pix_df)
            nuc_all.append(nuc_df)
            cov_all.append(cov_df)
            add_image_summary_slide(batch_ppt, plot_dir, fname)
            n_imp = int(nuc_df['imputed'].sum()) if 'imputed' in nuc_df.columns else 0
            imp_txt = f' +{n_imp} imputed' if n_imp else ''
            print(f'  ✓ {fname} ({nuc_df.shape[0]} nuclei @ {pixel_size_for(fname):.3f} µm/px{imp_txt})')
        except Exception as e:
            with open(failure_log, 'a') as f:
                f.write(f'{fname}\t{type(e).__name__}: {e}\n')
            print(f'  ✗ {fname} — {type(e).__name__}: {e}  (logged)')

    if pix_all:
        pix = pd.concat(pix_all, ignore_index=True)
        nuc = pd.concat(nuc_all, ignore_index=True)
        pix.to_csv(os.path.join(summary_root, 'combined_radial_profiles.csv'), index=False)
        nuc.to_csv(os.path.join(summary_root, 'combined_per_nucleus.csv'), index=False)

        ylabel = 'Normalized Intensity' if NORMALIZE_PROFILES else 'Intensity (a.u.)'

        # Cross-colony per-channel figures (per-pixel, classic style)
        for c, name in enumerate(CHANNEL_NAMES):
            fig, ax = plt.subplots(figsize=(8, 6))
            for img_name, grp in pix.groupby('Image'):
                ax.plot(grp['Radius'],
                        pd.Series(grp[f'C{c}'].values).interpolate(limit_direction='both'),
                        lw=1.5, alpha=0.8, label=clean_label(img_name))
            ax.set_title(f'Combined Radial Profiles - {name}')
            ax.set_xlabel('Distance from Center (µm)')
            ax.set_ylabel(f'{name} {ylabel}')
            ax.set_xlim(0, PLOT_MAX_UM)
            ax.grid(True, linestyle=':', alpha=0.7)
            ax.legend(fontsize=7, loc='lower right')
            fig.savefig(os.path.join(summary_root, f'combined_profiles_{name}.png'),
                        dpi=300, bbox_inches='tight')
            plt.close(fig)

        # Per-colony quantitative features + QC
        feat_rows = []
        exp_area_um2 = np.pi * (EXPECTED_NUCLEUS_DIAMETER_UM / 2.0) ** 2
        for img_name, grp in nuc.groupby('Image'):
            g_ok = grp[grp['size_ok']] if 'size_ok' in grp.columns else grp
            if 'imputed' in g_ok.columns:
                g_ok = g_ok[~g_ok['imputed']]   # features & stats: REAL nuclei only
            n_rej = int(len(grp) - len(g_ok))
            r_col = g_ok['distance_um'].quantile(0.99) if len(g_ok) else np.nan
            capacity = (NUCLEAR_PACKING_FRACTION * np.pi * r_col**2 / exp_area_um2
                        if np.isfinite(r_col) and r_col > 0 else np.nan)
            occupancy = len(g_ok) / capacity if capacity and np.isfinite(capacity) else np.nan
            gb = g_ok.copy()
            gb['bin'] = (gb['distance_um'] // BIN_WIDTH_UM) * BIN_WIDTH_UM
            cnts = gb.groupby('bin')[f'C{DAPI_IDX}'].count()
            good = cnts[cnts >= np.maximum(3, MIN_BIN_COVERAGE * cnts.median())].index
            dmeans = gb[gb['bin'].isin(good)].groupby('bin')[f'C{DAPI_IDX}'].mean()
            dapi_cv = float(dmeans.std() / dmeans.mean()) if len(dmeans) > 2 and dmeans.mean() else np.nan
            area_cv = float(g_ok['area_um2'].std() / g_ok['area_um2'].mean()) if len(g_ok) > 2 else np.nan
            pct_rej = 100.0 * n_rej / len(grp) if len(grp) else np.nan
            n_dir_gaps = np.nan
            if len(g_ok) > 50 and 'x_px' in g_ok.columns:
                ang = np.arctan2(g_ok['y_px'] - g_ok['y_px'].mean(),
                                 g_ok['x_px'] - g_ok['x_px'].mean())
                sec = ((ang + np.pi) / (2 * np.pi) * DIRECTIONAL_SECTORS).astype(int)
                sec = sec.clip(0, DIRECTIONAL_SECTORS - 1)
                gb2 = pd.DataFrame({'bin': (g_ok['distance_um'] // BIN_WIDTH_UM) * BIN_WIDTH_UM,
                                    'sec': sec.values})
                cov = gb2.groupby('bin')['sec'].nunique() / DIRECTIONAL_SECTORS
                cov = cov[cov.index >= 3 * BIN_WIDTH_UM]
                n_dir_gaps = int((cov < MIN_SECTOR_COVERAGE).sum())
            for c, name in enumerate(CHANNEL_NAMES):
                feats = extract_features(g_ok, f'C{c}')
                if feats is None:
                    continue
                feats.update({'Image': img_name, 'Channel': name,
                              'angular_cv': angular_cv(g_ok, f'C{c}'),
                              'n_rejected_size': n_rej,
                              'pct_rejected_size': pct_rej,
                              'area_cv': area_cv,
                              'n_directional_gap_bins': n_dir_gaps,
                              'occupancy_vs_capacity': occupancy,
                              'dapi_bin_cv': dapi_cv})
                feat_rows.append(feats)
        feat_df = pd.DataFrame(feat_rows)
        feat_df.to_csv(os.path.join(summary_root, 'colony_features.csv'), index=False)
        print('\nPer-colony features (mean ± SD across colonies):')
        for name in CHANNEL_NAMES:
            sub = feat_df[feat_df['Channel'] == name]
            if len(sub):
                print(f"  {name}: peak at {sub['peak_radius_um'].mean():.0f}±"
                      f"{sub['peak_radius_um'].std():.0f} µm, half-max boundary "
                      f"{sub['halfmax_radius_um'].mean():.0f}±{sub['halfmax_radius_um'].std():.0f} µm, "
                      f"center:edge {sub['center_edge_ratio'].mean():.2f}")
        high_cv = feat_df[feat_df['angular_cv'] > 0.25]['Image'].unique()
        if len(high_cv):
            print(f'  ⚠ QC: high angular asymmetry (CV>0.25) in: {list(high_cv)}')
        occ = feat_df.drop_duplicates('Image')
        over  = occ[occ['occupancy_vs_capacity'] > 1.2]['Image'].tolist()
        under = occ[occ['occupancy_vs_capacity'] < 0.3]['Image'].tolist()
        gaps  = occ[occ['dapi_bin_cv'] > 0.25]['Image'].tolist()
        if over:
            print(f'  ⚠ QC: nuclei exceed size-based capacity (over-segmentation or wrong '
                  f'EXPECTED_NUCLEUS_DIAMETER_UM?): {over}')
        if under:
            print(f'  ⚠ QC: nuclei far below capacity (sparse / under-segmentation?): {under}')
        if gaps:
            print(f'  ⚠ QC: non-uniform DAPI profile (cell-free zones?): {gaps}')
        segbad = occ[(occ['pct_rejected_size'] > 15) | (occ['area_cv'] > 0.6)]['Image'].tolist()
        dirbad = occ[occ['n_directional_gap_bins'] >= 3]['Image'].tolist()
        if segbad:
            print(f'  ⚠ QC: suspect segmentation (>15% size-rejected or area CV>0.6): {segbad}')
        if dirbad:
            print(f'  ⚠ QC: directional coverage gaps (>=3 bins missing sectors): {dirbad}')

        # Publication-style multi-panel figure (per-nucleus + LOWESS)
        nuc_ok = nuc[nuc['size_ok']] if 'size_ok' in nuc.columns else nuc
        # cap at MAX_RADIUS_UM: beyond the pattern radius live only arbitrary
        # off-colony objects (same rationale as the per-pixel NaN truncation)
        nuc_c = nuc_ok[nuc_ok['distance_um'] <= MAX_RADIUS_UM].copy()
        nuc_c['bin'] = (nuc_c['distance_um'] // BIN_WIDTH_UM) * BIN_WIDTH_UM + BIN_WIDTH_UM/2
        n_col = nuc_c['Image'].nunique()

        cov = pd.concat(cov_all, ignore_index=True)
        cov.to_csv(os.path.join(summary_root, 'bin_coverage.csv'), index=False)
        W_tech = cov.pivot_table(index='Image', columns='bin', values='coverage').clip(0, 1)

        # Packing weight: the pipeline reads each colony's marble-in-jar occupancy map
        # itself (the deep-dive panel J) — per radial bin, the mean over angular sectors of
        # min(1, observed/predicted). Perfectly packed -> 1; directional gaps pull the
        # weight down even when the bin's total count looks fine.
        A_NUC_UM2 = np.pi * (EXPECTED_NUCLEUS_DIAMETER_UM / 2.0) ** 2
        packs = []
        for img_name, g in nuc_c.groupby('Image'):
            cxn, cyn = g['x_px'].mean(), g['y_px'].mean()
            angn = np.arctan2(g['y_px'] - cyn, g['x_px'] - cxn)
            secn = np.clip(((angn + np.pi) / (2 * np.pi) * DIRECTIONAL_SECTORS).astype(int),
                           0, DIRECTIONAL_SECTORS - 1)
            cnt3 = g.groupby([g['bin'], secn]).size().unstack(fill_value=0)
            cnt3 = cnt3.reindex(columns=range(DIRECTIONAL_SECTORS), fill_value=0)
            binsp = cnt3.index.values.astype(float)
            predp = (NUCLEAR_PACKING_FRACTION * np.pi *
                     ((binsp + BIN_WIDTH_UM / 2) ** 2 - (binsp - BIN_WIDTH_UM / 2) ** 2)
                     / DIRECTIONAL_SECTORS / A_NUC_UM2)
            occp = np.clip(cnt3.values / predp[:, None], 0, 1)
            packs.append(pd.Series(occp.mean(axis=1), index=cnt3.index, name=img_name))
        W_pack = pd.DataFrame(packs)
        W_pack.index.name = 'Image'

        idxW = W_tech.index.union(W_pack.index)
        colsW = W_tech.columns.union(W_pack.columns)
        W_all = (W_tech.reindex(index=idxW, columns=colsW).fillna(0) *
                 W_pack.reindex(index=idxW, columns=colsW).fillna(0))
        W_all.round(3).to_csv(os.path.join(summary_root, 'bin_weights.csv'))
        pd.concat({'technical': W_tech, 'packing': W_pack},
                  names=['component']).round(3).to_csv(
            os.path.join(summary_root, 'bin_weight_components.csv'))

        pack_score = W_pack.mean(axis=1)
        print('\nColony packing scores (auto-read of occupancy maps): ' +
              '  '.join(f'{clean_label(i)}={v:.2f}' for i, v in pack_score.items()))
        low_pack = pack_score[pack_score < 0.5].index.tolist()
        if low_pack:
            print(f'  ↓ reduced weight in the group graph: {[clean_label(i) for i in low_pack]}')

        def colony_bootstrap_band(nuc_binned, col, n_boot=BOOTSTRAP_N):
            """Technically-weighted 95% CI of the binned mean, resampling COLONIES.

            Weight = technical coverage (segmented / DAPI+ area) x packing occupancy
            (sector-aware observed/predicted from the marble-in-jar model). Perfectly
            packed, well-segmented regions dominate; gappy or poorly covered regions
            contribute proportionally less. Zero-weight bins are interpolated across so
            exclusions never appear as sudden drops."""
            piv = nuc_binned.groupby(['Image', 'bin'])[col].mean().unstack('bin')
            if len(piv) < 4:
                return None
            w = W_all.reindex(index=piv.index, columns=piv.columns).fillna(0)
            P, W = piv.values, w.values
            W = np.where(np.isnan(P), 0.0, W)   # no observed mean -> no vote

            def wmean(rows):
                num = np.nansum(np.nan_to_num(P[rows]) * W[rows], axis=0)
                den = W[rows].sum(axis=0)
                return np.where(den > 0, num / den, np.nan)

            rng = np.random.default_rng(0)
            idx = rng.integers(0, len(piv), size=(n_boot, len(piv)))
            boots = np.array([wmean(r) for r in idx])
            lo, hi = np.nanpercentile(boots, [2.5, 97.5], axis=0)
            smooth = lambda a: pd.Series(a).interpolate(limit_direction='both').values
            return piv.columns.values, smooth(wmean(np.arange(len(piv)))), smooth(lo), smooth(hi)

        with mpl.rc_context(NATURE_RC):
            fig, axes = plt.subplots(1, len(CHANNEL_NAMES), figsize=(7.2, 2.4))
            for ax, (c, name) in zip(np.atleast_1d(axes), enumerate(CHANNEL_NAMES)):
                color = CHANNEL_COLORS[name]
                for img_name, grp in nuc_c.groupby('Image'):
                    grp = grp.sort_values('distance_um')
                    if len(grp) < 30:
                        continue
                    sm = lowess(grp[f'C{c}'].values, grp['distance_um'].values,
                                frac=LOWESS_FRAC, return_sorted=True)
                    ax.plot(sm[:, 0], sm[:, 1], color=color, lw=0.5, alpha=0.25)
                band = colony_bootstrap_band(nuc_c, f'C{c}')
                if band is not None:
                    bx, bmean, blo, bhi = band
                    ax.fill_between(bx, blo, bhi, color=color, alpha=0.25, linewidth=0)
                    ax.plot(bx, bmean, color=color, lw=1.8)
                else:
                    by = nuc_c.groupby('bin')[f'C{c}'].agg(['mean', 'sem']).reset_index()
                    ax.fill_between(by['bin'], by['mean']-by['sem'], by['mean']+by['sem'],
                                    color=color, alpha=0.25, linewidth=0)
                    ax.plot(by['bin'], by['mean'], color=color, lw=1.8)
                ax.set_xlim(0, PLOT_MAX_UM)
                ax.set_xlabel('Distance from center (µm)')
                ax.set_ylabel(f'{name} (a.u.)')
                ax.set_title(name)
                ax.text(0.03, 0.97, f'n = {n_col} colonies\n{len(nuc_c):,} nuclei',
                        transform=ax.transAxes, va='top', ha='left',
                        fontsize=6.5, color='#333333')
            for ax, letter in zip(np.atleast_1d(axes), 'abcdefg'):
                ax.text(-0.20, 1.02, letter, transform=ax.transAxes,
                        fontsize=11, fontweight='bold', va='top', ha='left')
            plt.tight_layout()
            fig.savefig(os.path.join(summary_root, 'publication_figure.png'),
                        dpi=300, bbox_inches='tight', facecolor='white')
            fig.savefig(os.path.join(summary_root, 'publication_figure.pdf'),
                        bbox_inches='tight', facecolor='white')
            plt.close(fig)

        # Fraction of positive nuclei vs radius (Otsu per colony; Gros et al. 2026 sparse-
        # marker approach — their FoxA2 analysis)
        if POSITIVE_FRACTION:
            fig, axesP = plt.subplots(1, len(CHANNEL_NAMES), figsize=(5*len(CHANNEL_NAMES), 4))
            for ci, name in enumerate(CHANNEL_NAMES):
                ax = np.atleast_1d(axesP)[ci]
                agg_rows = []
                for img_name, grp in nuc_c.groupby('Image'):
                    vals = grp[f'C{ci}'].dropna()
                    if len(vals) < 50:
                        continue
                    thr = threshold_otsu(vals.values)
                    g = grp.copy(); g['pos'] = g[f'C{ci}'] >= thr
                    frac = g.groupby('bin')['pos'].mean()
                    ax.plot(frac.index, frac.values, lw=1, alpha=0.35,
                            color=CHANNEL_COLORS[name])
                    agg_rows.append(g[['bin', 'pos']])
                if agg_rows:
                    agg = pd.concat(agg_rows).groupby('bin')['pos'].mean()
                    ax.plot(agg.index, agg.values, lw=2.5, color=CHANNEL_COLORS[name])
                ax.set_xlim(0, PLOT_MAX_UM); ax.set_ylim(0, 1)
                ax.set_xlabel('Distance from Center (µm)')
                ax.set_ylabel(f'Fraction {name}+ nuclei')
                ax.set_title(f'{name}+ fraction (Otsu per colony)')
            plt.tight_layout()
            fig.savefig(os.path.join(summary_root, 'positive_fraction_profiles.png'),
                        dpi=300, bbox_inches='tight')
            plt.close(fig)

        # Per-nucleus pairwise co-expression histograms (Gros et al. 2026 Fig. 5f)
        if COEXPRESSION_PLOTS:
            pairs = list(itertools.combinations(range(len(CHANNEL_NAMES)), 2))
            fig, axesX = plt.subplots(1, len(pairs), figsize=(5.2*len(pairs), 4.2))
            for ax, (a, b) in zip(np.atleast_1d(axesX), pairs):
                xa, xb = nuc_ok[f'C{a}'].values, nuc_ok[f'C{b}'].values
                ok = np.isfinite(xa) & np.isfinite(xb)
                h = ax.hist2d(xa[ok], xb[ok], bins=80, cmap='inferno',
                              norm=mpl.colors.LogNorm())
                ta, tb = threshold_otsu(xa[ok]), threshold_otsu(xb[ok])
                ax.axvline(ta, color='w', lw=0.8, ls='--')
                ax.axhline(tb, color='w', lw=0.8, ls='--')
                quads = {(0.97, 0.97): ((xa[ok] >= ta) & (xb[ok] >= tb)).mean(),
                         (0.97, 0.05): ((xa[ok] >= ta) & (xb[ok] <  tb)).mean(),
                         (0.03, 0.97): ((xa[ok] <  ta) & (xb[ok] >= tb)).mean(),
                         (0.03, 0.05): ((xa[ok] <  ta) & (xb[ok] <  tb)).mean()}
                for (fx, fy), frac in quads.items():
                    ax.text(fx, fy, f'{frac*100:.0f}%', transform=ax.transAxes,
                            ha='right' if fx > 0.5 else 'left',
                            va='top' if fy > 0.5 else 'bottom',
                            color='w', fontsize=8, fontweight='bold')
                r = np.corrcoef(xa[ok], xb[ok])[0, 1]
                ax.set_xlabel(f'{CHANNEL_NAMES[a]} (a.u.)')
                ax.set_ylabel(f'{CHANNEL_NAMES[b]} (a.u.)')
                ax.set_title(f'{CHANNEL_NAMES[a]} vs {CHANNEL_NAMES[b]}  (r = {r:.2f})')
                fig.colorbar(h[3], ax=ax, label='# nuclei')
            plt.tight_layout()
            fig.savefig(os.path.join(summary_root, 'coexpression_per_nucleus.png'),
                        dpi=300, bbox_inches='tight')
            plt.close(fig)

        # Nuclear morphometrics vs radius (nuclei as deformation proxies)
        fig, axesM = plt.subplots(1, 2, figsize=(11, 4))
        al = nuc_c.dropna(subset=['cos2_radial'])
        if len(al):
            byA = al.groupby('bin')['cos2_radial'].agg(['mean', 'sem']).reset_index()
            axesM[0].fill_between(byA['bin'], byA['mean']-byA['sem'], byA['mean']+byA['sem'],
                                  alpha=0.3, color='#7E6148')
            axesM[0].plot(byA['bin'], byA['mean'], color='#7E6148', lw=2)
        axesM[0].axhline(0.5, color='gray', ls=':', lw=1)
        axesM[0].set_ylim(0, 1); axesM[0].set_xlim(0, PLOT_MAX_UM)
        axesM[0].set_xlabel('Distance from Center (µm)')
        axesM[0].set_ylabel('cos²(major axis, radial)')
        axesM[0].set_title('Nuclear radial alignment (0.5 = random)')
        ecd = nuc_c.dropna(subset=['eccentricity'])
        if len(ecd):
            byE = ecd.groupby('bin')['eccentricity'].agg(['mean', 'sem']).reset_index()
            axesM[1].fill_between(byE['bin'], byE['mean']-byE['sem'], byE['mean']+byE['sem'],
                                  alpha=0.3, color='#4DBBD5')
            axesM[1].plot(byE['bin'], byE['mean'], color='#4DBBD5', lw=2)
        axesM[1].set_xlim(0, PLOT_MAX_UM)
        axesM[1].set_xlabel('Distance from Center (µm)')
        axesM[1].set_ylabel('Eccentricity')
        axesM[1].set_title('Nuclear elongation vs radius')
        plt.tight_layout()
        fig.savefig(os.path.join(summary_root, 'nuclear_morphometrics.png'),
                    dpi=300, bbox_inches='tight')
        plt.close(fig)

        batch_ppt.save(os.path.join(ppt_root, 'batch_report.pptx'))
        print(f'\nDone. Summary outputs in {summary_root}')
        print('  combined_radial_profiles.csv  (per-pixel)')
        print('  combined_per_nucleus.csv      (per-nucleus)')
        print('  publication_figure.png / .pdf')
        build_qc_report(BASE_DIR)
        print('  qc_report.html                (open in a browser)')
    else:
        print('No images processed successfully — see failed_images.log')


## Interactive GUI inspection (napari)

Same ecosystem as the Tapenade paper's plugins — napari is already in the environment.
`inspect_in_napari(path)` opens one image with all raw channels, the enhanced DAPI, the
nucleus labels, and the computed colony center. Toggle layers, adjust contrast, and zoom
to judge segmentation quality directly. (For their full preprocessing GUI, install
`napari-tapenade-processing`.)


In [ ]:
def inspect_in_napari(tif_path):
    """Open one image + its segmentation interactively in napari."""
    import napari
    img_enh, img_raw = load_and_preprocess(tif_path)
    labels, mask, center = segment_nuclei(img_enh)
    v = napari.Viewer(title=os.path.basename(tif_path))
    layer_colors = ['green', 'blue', 'red', 'magenta']
    for c in range(img_raw.shape[0]):
        v.add_image(img_raw[c], name=f'raw {CHANNEL_NAMES[c]}',
                    colormap=layer_colors[c % len(layer_colors)],
                    blending='additive', visible=(c == DAPI_IDX))
    v.add_image(img_enh[DAPI_IDX], name='DAPI enhanced', visible=False)
    v.add_labels(labels, name=f'nuclei ({int(labels.max())})')
    v.add_points(np.array([[center[1], center[0]]]), name='colony center',
                 size=20, face_color='yellow', symbol='cross')
    napari.run()
    return v

# Example — opens a GUI window (run locally, not headless):
# inspect_in_napari(os.path.join(BASE_DIR, 'my_image.tif'))


## Group comparison — mean profile ± SD per experimental group

Compare genotypes/treatments (e.g. RUES2 vs 72CAG vs HTTKO, ± drug): each colony's profile
is optionally normalized to its own max, then averaged within its group; the band is ±1 SD
(or SEM) across colonies. Groups are assigned by filename substring. One subplot per panel.


In [ ]:
# Map panel title -> {group label: filename substring}. Edit for your experiment, e.g.:
# GROUP_PANELS = {
#     'Activin':      {'RUES2 ACTIVIN': 'RUES2+ActA', '72CAG ACTIVIN': '72CAG_Act',
#                      'HTTKO ACTIVIN': 'HTTKO_Act'},
#     'Activin+BRD9': {'RUES2 ACTIVIN+BRD9': 'RUES2_BRD9', '72CAG ACTIVIN+BRD9': '72CAG_BRD9',
#                      'HTTKO ACTIVIN+BRD9': 'HTTKO_BRD9'},
# }
GROUP_PANELS    = {'All colonies': {'All': ''}}
GROUP_CHANNEL   = 'C2'    # channel to compare (C2 = SMAD2)
GROUP_NORMALIZE = True    # normalize each colony to its own max before averaging
GROUP_BAND      = 'sd'    # 'sd' (classic "± 1 SD by Group") or 'sem'
GROUP_Y_TICK    = 0.2     # y-axis tick step — few coarse divisions; axis stays cropped
                          # to the data range rather than showing the full scale
GROUP_ZOOM_BURST = False  # optional: split x-axis (compress interior, expand ramp)
GROUP_BASELINE   = False  # per-COLONY shape display only: plateau -> 0, peak -> 1.
                          # Keep OFF for group comparisons — plateau-level differences
                          # between genotypes/treatments are real signal, and baseline
                          # rescaling would erase them.


def plot_group_comparison(pix, panels=None, channel=None, normalize=None, band=None,
                          out_path=None, max_r=None, zoom_burst=None, burst_start=None,
                          y_tick_step=None, baseline=None):
    """Mean per-pixel profile ± band per experimental group.

    With zoom_burst the x-axis is split at the ramp onset: the interior is compressed,
    the rise region expanded (broken-axis marks at the join), y-ticks are fine-grained,
    and y-limits hug the data — so the radial increase reads at a glance."""
    from matplotlib.ticker import MultipleLocator
    panels = panels or GROUP_PANELS
    channel = channel or GROUP_CHANNEL
    normalize = GROUP_NORMALIZE if normalize is None else normalize
    band = band or GROUP_BAND
    max_r = max_r or MAX_RADIUS_UM
    zoom_burst = GROUP_ZOOM_BURST if zoom_burst is None else zoom_burst
    baseline = GROUP_BASELINE if baseline is None else baseline
    y_tick_step = y_tick_step or GROUP_Y_TICK
    ylab = ('Normalized \u0394Intensity (plateau \u2192 0)' if baseline
            else 'Normalized Intensity' if normalize else 'Intensity (a.u.)')
    palette = ['#E64B35', '#4DBBD5', '#00A087', '#3C5488', '#F39B7F', '#8491B4']

    # ---- gather group statistics first (needed for burst detection / y-limits) ----
    panel_stats = {}
    for panel_name, groups in panels.items():
        stats = []
        for gi, (label, pattern) in enumerate(groups.items()):
            imgs = [im for im in pix['Image'].unique() if pattern in im]
            if not imgs:
                print(f'  (no images match {label!r} pattern {pattern!r})')
                continue
            curves = []
            for im in imgs:
                g = pix[pix['Image'] == im].sort_values('Radius')
                g = g[g['Radius'] <= max_r]
                y = pd.Series(g[channel].values).interpolate(limit_direction='both')
                if normalize and y.max() > 0:
                    y = y / y.max()
                if baseline:
                    inner = y.values[:max(10, int(len(y) * 0.4))]
                    b = np.nanpercentile(inner, 20)   # interior plateau level
                    rng = np.nanmax(y.values) - b
                    if rng > 0:
                        y = (y - b) / rng             # plateau -> 0, peak -> 1
                curves.append(pd.Series(y.values, index=g['Radius'].round(1)))
            mat = pd.concat(curves, axis=1)
            mean = mat.mean(axis=1)
            spread = mat.std(axis=1) if band == 'sd' else mat.sem(axis=1)
            stats.append((f'{label} (n={len(imgs)})', mean, spread,
                          palette[gi % len(palette)]))
        panel_stats[panel_name] = stats

    with mpl.rc_context(NATURE_RC):
        n_p = len(panels)
        if zoom_burst:
            fig = plt.figure(figsize=(5.8 * n_p, 3.6))
            gs = fig.add_gridspec(1, 2 * n_p, width_ratios=[1, 1.8] * n_p, wspace=0.06)
        else:
            fig, flat_axes = plt.subplots(1, n_p, figsize=(4.6 * n_p, 3.4), squeeze=False)

        for pi, (panel_name, stats) in enumerate(panel_stats.items()):
            if not stats:
                continue
            # auto-detect ramp onset from the pooled mean curve
            pooled = pd.concat([m for _, m, _, _ in stats]).groupby(level=0).mean()
            lo, hi = pooled.min(), pooled.max()
            cross = pooled[pooled >= lo + 0.25 * (hi - lo)]
            auto_burst = float(cross.index[0]) if len(cross) else max_r * 0.5
            split = burst_start or max(50.0, min((auto_burst // 25) * 25 - 25, max_r - 75))

            ymin = min(float((m - s).min()) for _, m, s, _ in stats)
            ymax = max(float((m + s).max()) for _, m, s, _ in stats)
            ylo = np.floor((ymin - 0.02) / y_tick_step) * y_tick_step
            yhi = np.ceil((ymax + 0.02) / y_tick_step) * y_tick_step

            if zoom_burst:
                ax_l = fig.add_subplot(gs[0, 2 * pi])
                ax_r = fig.add_subplot(gs[0, 2 * pi + 1], sharey=ax_l)
                pair = (ax_l, ax_r)
            else:
                pair = (flat_axes[0][pi],)

            for ax in pair:
                for label, mean, spread, color in stats:
                    ax.plot(mean.index, mean.values, color=color, lw=1.8, label=label)
                    ax.fill_between(mean.index, mean - spread, mean + spread,
                                    color=color, alpha=0.25, linewidth=0)
                ax.yaxis.set_major_locator(MultipleLocator(y_tick_step))
                ax.grid(True, axis='y', linestyle=':', alpha=0.5)
                ax.set_ylim(ylo, yhi)

            if zoom_burst:
                ax_l.set_xlim(0, split)
                ax_r.set_xlim(split, max_r + 10)
                ax_l.spines['right'].set_visible(False)
                ax_r.spines['left'].set_visible(False)
                ax_r.tick_params(labelleft=False, left=False)
                d = 0.5   # broken-axis marks at the join
                kw = dict(marker=[(-1, -d), (1, d)], markersize=8, linestyle='none',
                          color='k', mec='k', mew=1, clip_on=False)
                ax_l.plot([1, 1], [0, 1], transform=ax_l.transAxes, **kw)
                ax_r.plot([0, 0], [0, 1], transform=ax_r.transAxes, **kw)
                ax_l.set_ylabel(ylab)
                ax_r.set_xlabel('Distance (µm)')
                ax_r.set_title(f'{panel_name}   (x expanded beyond {split:.0f} µm)')
                ax_r.legend(title='Group (n)', fontsize=6.5, title_fontsize=6.5,
                            loc='lower right')
            else:
                ax = pair[0]
                ax.set_xlim(0, max_r + 10)
                ax.set_xlabel('Distance (µm)')
                ax.set_ylabel(ylab)
                ax.set_title(panel_name)
                ax.legend(title='Group (n)', fontsize=6.5, title_fontsize=6.5,
                          loc='lower left')

        plt.tight_layout()
        if out_path:
            fig.savefig(out_path, dpi=300, bbox_inches='tight', facecolor='white')
            fig.savefig(out_path.replace('.png', '.pdf'), bbox_inches='tight',
                        facecolor='white')
        plt.show()
    return fig


# Example (after a batch run):
# pix = pd.read_csv(os.path.join(BASE_DIR, 'summary', 'combined_radial_profiles.csv'))
# plot_group_comparison(pix, out_path=os.path.join(BASE_DIR, 'summary',
#                                                  'group_comparison_SMAD2.png'))


---
## Future work: 3D gastruloid segmentation with the pretrained tapenade StarDist3D

The Tapenade group provides their custom **StarDist3D** model (`tapenade_stardist`), trained
on 4,414 annotated gastruloid nuclei (F1 = 85±3%, constant across >200 µm depth). The cells
below download it from Zenodo (12.6 MB, cached locally), load it, and provide a 3D
per-nucleus table that mirrors this pipeline's 2D analysis — with **distance to the sample
border** (the paper's radial coordinate for variable-size 3D samples) instead of distance
to a 2D colony center.

**Model constraints (from their readme):** isotropic voxels, nuclei ≈ 15 px diameter
(so ≈ 1 µm/voxel for typical 10–15 µm nuclei), intensity normalized to [0, 1].
Set `Z_STEP_UM` to your acquisition's z-spacing.

These cells are self-contained and do not run in the 2D batch above.


In [ ]:
TAPENADE_ZENODO_API = 'https://zenodo.org/api/records/14748083'
TAPENADE_MODEL_ZIP  = 'stardist_tapenade_model.zip'   # 12.6 MB (ignore the 9.4 GB data zip)


def load_tapenade_stardist3d(model_root='models'):
    """Download (once, cached) and load the pretrained tapenade StarDist3D model
    (Gros et al., eLife 2026; Zenodo 14748083)."""
    import urllib.request, zipfile
    os.makedirs(model_root, exist_ok=True)
    basedir = os.path.join(model_root, 'stardist_tapenade_model')
    if not os.path.isdir(os.path.join(basedir, 'tapenade_stardist')):
        url = f'{TAPENADE_ZENODO_API}/files/{TAPENADE_MODEL_ZIP}/content'
        zip_path = os.path.join(model_root, TAPENADE_MODEL_ZIP)
        print(f'Downloading {TAPENADE_MODEL_ZIP} from Zenodo…')
        urllib.request.urlretrieve(url, zip_path)
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(model_root)
        os.remove(zip_path)
    from stardist.models import StarDist3D
    return StarDist3D(None, name='tapenade_stardist', basedir=basedir)


def segment_nuclei_3d(dapi_zyx, z_step_um, xy_um_per_px=SCALE_UM_PER_PX,
                      target_um_per_vox=1.0, model=None):
    """Segment nuclei in a 3D (Z, Y, X) stack with the tapenade StarDist3D model.

    Rescales to isotropic voxels of `target_um_per_vox` (their model expects nuclei of
    ~15 px diameter, i.e. ~1 um/voxel for 10-15 um nuclei), normalizes, predicts, and
    returns (labels_isotropic, voxel_size_um)."""
    from scipy.ndimage import zoom as _zoom
    if model is None:
        model = load_tapenade_stardist3d()
    factors = (z_step_um / target_um_per_vox,
               xy_um_per_px / target_um_per_vox,
               xy_um_per_px / target_um_per_vox)
    iso = _zoom(dapi_zyx.astype(np.float32), factors, order=1)
    norm = global_contrast_enhancement(iso, perc_low=1, perc_high=99)
    labels, _ = model.predict_instances(norm)
    return labels, target_um_per_vox


In [ ]:
def per_nucleus_table_3d(channels_iso, labels, voxel_um):
    """3D analogue of per_nucleus_table for gastruloids.

    channels_iso : (C, Z, Y, X) intensity channels at the SAME isotropic voxel size
                   as `labels` (rescale them with the same zoom factors).
    labels       : 3D label image from segment_nuclei_3d.
    Returns one row per nucleus: centroid, volume, distance to the sample border
    (the Tapenade paper's radial coordinate), and per-channel mean intensities."""
    from scipy.ndimage import distance_transform_edt, binary_fill_holes, binary_closing

    ids = np.arange(1, int(labels.max()) + 1)
    ones = np.ones(labels.shape, dtype=np.float32)
    coms = ndimage.center_of_mass(ones, labels, ids)
    vol_vox = ndimage.sum_labels(ones, labels, ids)

    # Sample mask: closed + filled union of nuclei; EDT gives distance to border
    sample = binary_fill_holes(binary_closing(labels > 0, np.ones((5, 5, 5))))
    edt = distance_transform_edt(sample) * voxel_um

    zs = np.array([c[0] for c in coms]); ys = np.array([c[1] for c in coms])
    xs = np.array([c[2] for c in coms])
    zi = np.clip(zs.round().astype(int), 0, labels.shape[0]-1)
    yi = np.clip(ys.round().astype(int), 0, labels.shape[1]-1)
    xi = np.clip(xs.round().astype(int), 0, labels.shape[2]-1)

    out = {
        'nucleus_id': ids, 'z_vox': zs, 'y_vox': ys, 'x_vox': xs,
        'volume_um3': vol_vox * voxel_um**3,
        'dist_to_border_um': edt[zi, yi, xi],
    }
    for c in range(channels_iso.shape[0]):
        out[f'C{c}'] = ndimage.mean(channels_iso[c], labels, ids)
    return pd.DataFrame(out)


# --- Example usage on a (Z, C, Y, X) two-photon stack ------------------------------
# from scipy.ndimage import zoom
# Z_STEP_UM = 1.0                                    # <- your z spacing!
# stack = tifffile.imread('my_gastruloid.tif')       # (Z, C, Y, X)
# labels3d, vox = segment_nuclei_3d(stack[:, DAPI_IDX], Z_STEP_UM)
# factors = (Z_STEP_UM / vox, SCALE_UM_PER_PX / vox, SCALE_UM_PER_PX / vox)
# chans = np.stack([zoom(stack[:, c].astype(np.float32), factors, order=1)
#                   for c in range(stack.shape[1])])
# nuc3d = per_nucleus_table_3d(chans, labels3d, vox)
# nuc3d.to_csv('gastruloid_nuclei_3d.csv', index=False)
# # then profile any channel vs nuc3d['dist_to_border_um'] exactly like the 2D LOWESS plots
